# Teoria da Computabilidade — Projetos 04 e 05

Notebook único, sem dependências entre arquivos: todas as funções do
**Projeto 04** (Codificação de Gödel) são definidas primeiro e reutilizadas
diretamente pelo **Projeto 05** (Simulador Turing-Completo via μ-Recursão)
mais abaixo, no mesmo kernel — sem `%run`, sem `import`, sem depender de
`nbformat` ou de qualquer outro arquivo auxiliar.

**Como usar:** basta subir este único arquivo `.ipynb` e rodar as células
em ordem, de cima para baixo. Funciona em Colab, VS Code ou JupyterLab.

---

## Sumário
1. Projeto 04 — Codificação de Gödel & A Função Beta
2. Projeto 05 — O Simulador de Turing-Completo via Funções μ-Recursivas

---
# Parte 1 — Projeto 04: Codificação de Gödel & A Função Beta

**Foco:** Propriedades de Fechamento, Representação de Estruturas de Dados e Codificação Semântica em FRPs (Funções Recursivas Primitivas).

Implementa:

1. **Módulo de Codificação** — `codificar(lista) -> N`, via produto de potências de primos.
2. **Módulo de Decodificação** — `tamanho(N)` e `obter_elemento(N, i)`, usando apenas minimização **limitada** (permanece FRP).
3. **Aplicação Prática** — Sequência de Fibonacci com memória empacotada em um único número natural N.

> Todas as funções de decodificação usam apenas SUCESSOR, PROJEÇÃO, COMPOSIÇÃO e MINIMIZAÇÃO LIMITADA — nenhuma busca aqui é ilimitada, pois todo limite superior é calculável a partir do próprio N.

## 0. Geração de primos (função auxiliar, FRP: busca sempre limitada)

In [1]:
from itertools import count


def _eh_primo(n: int) -> bool:
    """Teste de primalidade por tentativa de divisão — busca limitada a sqrt(n)."""
    if n < 2:
        return False
    if n in (2, 3):
        return True
    if n % 2 == 0:
        return False
    for d in range(3, int(n ** 0.5) + 1, 2):
        if n % d == 0:
            return False
    return True


def primo(i: int) -> int:
    """
    Retorna o i-ésimo primo (indexado a partir de i=1 -> 2, i=2 -> 3, ...).

    A busca é limitada pelo Postulado de Bertrand (existe sempre um primo
    entre k e 2k), então o número de candidatos testados é computável a
    priori a partir de i — permanece FRP.
    """
    if i < 1:
        raise ValueError("índice de primo deve ser >= 1")
    encontrados = 0
    for candidato in count(2):
        if _eh_primo(candidato):
            encontrados += 1
            if encontrados == i:
                return candidato

## 1. Módulo de Codificação: lista `[x1, ..., xk]` → N

In [2]:
def codificar(lista: list[int]) -> int:
    """
    N = 2^(x1+1) * 3^(x2+1) * 5^(x3+1) * ... * p_k^(xk+1)

    Codifica uma lista de inteiros não-negativos em um único número
    natural via o Teorema Fundamental da Aritmética (fatoração única).

    Nota sobre o deslocamento (+1): se um xi pudesse ser 0, o primo
    correspondente teria expoente 0 e, portanto, NÃO dividiria N — tornando
    impossível distinguir "elemento com valor 0" de "lista terminou aqui"
    usando apenas divisibilidade (que é o teste que tamanho() precisa, para
    permanecer FRP). Deslocar cada expoente em +1 garante que todo primo
    p_1..p_k sempre divide N pelo menos uma vez, preservando a
    decodificabilidade sem sair de FRP.
    """
    N = 1
    for i, x in enumerate(lista, start=1):
        N *= primo(i) ** (x + 1)
    return N

## 2. Módulo de Decodificação (estritamente FRP)

In [3]:
def _expoente_de(N: int, p: int) -> int:
    """
    Extrai o expoente de um primo p na fatoração de N.

    FRP porque o expoente máximo possível é log_p(N), um teto calculável
    diretamente de N — a busca (divisões sucessivas) é, portanto, limitada.
    """
    if N == 0:
        return 0
    expoente = 0
    limite = N.bit_length() + 1  # cota superior segura para o expoente
    for _ in range(limite):
        if N % p == 0:
            N //= p
            expoente += 1
        else:
            break
    return expoente

In [4]:
def tamanho(N: int) -> int:
    """
    tamanho(N): quantidade de elementos armazenados em N.

    Estratégia FRP: percorre primos p_1, p_2, ... testando se p_i divide N.
    Graças ao deslocamento (+1) em codificar(), todo primo realmente usado
    na lista sempre divide N pelo menos uma vez — então o primeiro primo
    que NÃO divide N marca o fim confiável da lista. A busca é limitada
    porque nenhum primo maior que N pode dividir N, então o número de
    primos a testar é, no pior caso, limitado por N.
    """
    if N <= 1:
        return 0
    i = 1
    while True:
        p = primo(i)
        if p > N:
            return i - 1
        if N % p != 0:
            return i - 1
        i += 1

In [5]:
def obter_elemento(N: int, i: int) -> int:
    """
    obter_elemento(N, i): retorna o i-ésimo elemento armazenado em N,
    isto é, o expoente do i-ésimo primo na fatoração de N, menos o
    deslocamento (+1) aplicado em codificar().

    Usa minimização LIMITADA: o expoente procurado nunca excede
    log_2(N), cota calculável a partir do próprio N.
    """
    p_i = primo(i)
    return _expoente_de(N, p_i) - 1

## 3. Aplicação Prática: Fibonacci com memória empacotada em um único N

In [6]:
def fibonacci_memo(n: int) -> int:
    """
    Calcula F(n) mantendo TODO o histórico de chamadas anteriores
    empacotado em um único número natural N (codificação de Gödel),
    demonstrando que uma FRP pode simular estado/memória sem sair de ℕ.

    N codifica a lista [F(0), F(1), ..., F(k)] computada até o momento.
    A cada passo, decodifica os dois últimos valores via obter_elemento,
    calcula o próximo e recodifica a lista estendida.
    """
    if n == 0:
        return 0
    if n == 1:
        return 1

    # Estado inicial: N codifica [F(0), F(1)] = [0, 1]
    N = codificar([0, 1])

    for k in range(2, n + 1):
        tam = tamanho(N)
        f_k_menos_2 = obter_elemento(N, tam - 1)  # F(k-2)
        f_k_menos_1 = obter_elemento(N, tam)      # F(k-1)
        f_k = f_k_menos_2 + f_k_menos_1

        # Reconstrói a lista completa a partir de N e adiciona o novo termo
        historico = [obter_elemento(N, j) for j in range(1, tam + 1)]
        historico.append(f_k)
        N = codificar(historico)

    return obter_elemento(N, tamanho(N))

### Demonstração — Projeto 04

**1. Codificação de `[3, 1, 4]`**

In [7]:
lista_exemplo = [3, 1, 4]
N = codificar(lista_exemplo)
print(f"Lista original : {lista_exemplo}")
print(f"N = 2^3 * 3^1 * 5^4 (com deslocamento +1) = {N}")

Lista original : [3, 1, 4]
N = 2^3 * 3^1 * 5^4 (com deslocamento +1) = 450000


**2. Decodificação de N usando apenas funções primitivas**

In [8]:
print(f"tamanho(N) = {tamanho(N)}  (esperado: {len(lista_exemplo)})")
for i in range(1, tamanho(N) + 1):
    print(f"obter_elemento(N, {i}) = {obter_elemento(N, i)}  "
          f"(esperado: {lista_exemplo[i-1]})")

tamanho(N) = 3  (esperado: 3)
obter_elemento(N, 1) = 3  (esperado: 3)
obter_elemento(N, 2) = 1  (esperado: 1)
obter_elemento(N, 3) = 4  (esperado: 4)


**3. Fibonacci com memória empacotada em um único N**

In [9]:
for n in range(10):
    print(f"F({n}) = {fibonacci_memo(n)}")

F(0) = 0
F(1) = 1
F(2) = 1
F(3) = 2
F(4) = 3
F(5) = 5
F(6) = 8
F(7) = 13
F(8) = 21
F(9) = 34


---
# Parte 2 — Projeto 05: O Simulador de Turing-Completo via Funções μ-Recursivas

**Foco:** Teorema da Normalização de Kleene, Tese de Church-Turing e Minimização Única.

Reaproveita `codificar`, `tamanho` e `obter_elemento` **já definidos na Parte 1
acima, no mesmo kernel** — não há import nem `%run` aqui, pois este é o mesmo
notebook. Reaproveita a codificação de Gödel em duas camadas:

1. a fita (lista de símbolos) → um único natural `T`
2. a configuração `(estado, cabeça, T)` → um único natural `S`

Implementa:

1. **Representador do Estado da Computação** → `S ∈ ℕ`
2. **Predicado de Transição Primitivo** → `T(e, S, y)` — **FRP**
3. **Laço do Interpretador Universal** → `μy [...]` — **NÃO-FRP**
4. **Aplicações de Teste** → máquina que para vs. loop infinito

## 1. Representador do Estado da Computação: `(estado, posição, fita) → S ∈ ℕ`

In [10]:
def codificar_configuracao(estado: int, posicao: int, fita: list[int]) -> int:
    """
    Empacota uma configuração instantânea (q, h, fita) em um único S ∈ N.

    Reaproveita codificar() do Projeto 04 DUAS vezes em camadas:
      T = codificar(fita)             -> a fita vira um natural
      S = codificar([q, h, T])        -> a tripla (q, h, T) vira um natural
    """
    T_fita = codificar(fita)
    return codificar([estado, posicao, T_fita])


def decodificar_configuracao(S: int) -> tuple[int, int, list[int]]:
    """Operação inversa: extrai (estado, posição, fita) a partir de S."""
    estado = obter_elemento(S, 1)
    posicao = obter_elemento(S, 2)
    T_fita = obter_elemento(S, 3)
    tam = tamanho(T_fita)
    fita = [obter_elemento(T_fita, i) for i in range(1, tam + 1)] if tam > 0 else []
    return estado, posicao, fita

## 2. Núcleo de transição (bounded — usado tanto por T quanto por U)

In [11]:
SIMBOLO_BRANCO = 0  # convenção: 0 representa a célula em branco


def passo(S: int, delta: dict) -> int | None:
    """
    Aplica UMA transição a partir da configuração codificada S.

    delta: dict{(estado, simbolo): (novo_estado, novo_simbolo, movimento)}
           movimento ∈ {-1, 0, +1}  (esquerda, permanece, direita)

    Retorna a nova configuração S' codificada, ou None se não existe
    transição para (estado, símbolo) — convenção adotada para "a máquina
    parou". Esta função é inteiramente limitada/FRP: decodifica S
    (bounded), consulta a tabela delta (O(1), bounded) e recodifica
    (bounded) — nenhuma parte desta função busca sem cota.
    """
    estado, posicao, fita = decodificar_configuracao(S)

    simbolo_atual = fita[posicao] if 0 <= posicao < len(fita) else SIMBOLO_BRANCO

    regra = delta.get((estado, simbolo_atual))
    if regra is None:
        return None  # nenhuma transição aplicável -> máquina parada

    novo_estado, novo_simbolo, movimento = regra

    # garante que a fita tenha a célula 'posicao' antes de escrever nela
    while len(fita) <= posicao:
        fita.append(SIMBOLO_BRANCO)
    fita[posicao] = novo_simbolo

    nova_posicao = posicao + movimento
    if nova_posicao < 0:
        # fita "infinita à esquerda": insere branco no início e realinha
        fita.insert(0, SIMBOLO_BRANCO)
        nova_posicao = 0

    return codificar_configuracao(novo_estado, nova_posicao, fita)

## 2b. Predicado de Transição Primitivo `T(e, S0, y)`

In [12]:
def T(delta: dict, S0: int, y: int) -> bool:
    """
    T(e, S0, y): verifica se a máquina 'delta', partindo de S0, executa
    exatamente (y - 1) transições válidas e então PARA no y-ésimo passo
    (isto é, passo() aplicado à (y-1)-ésima configuração retorna None).

    Para um y FIXO, esta verificação percorre no máximo y configurações
    — um laço de tamanho conhecido a priori. Logo, T é estritamente FRP:
    usa apenas minimização LIMITADA (limitada por y, que já é dado).
    Isso é exatamente o predicado de Kleene do Teorema da Forma Normal.
    """
    if y < 1:
        return False

    S = S0
    for _ in range(y - 1):
        proxima = passo(S, delta)
        if proxima is None:
            return False  # já havia parado antes do passo y
        S = proxima

    return passo(S, delta) is None

## 2c. Função de extração `U(e, S0, y)`

In [13]:
def U(delta: dict, S0: int, y: int) -> list[int]:
    """
    U(y): dado que T(e, S0, y) é verdadeiro (a máquina para no passo y),
    decodifica e retorna o conteúdo final da fita.

    Reconstrói a configuração final aplicando (y - 1) transições válidas
    a partir de S0 — a mesma quantidade de passos verificada por T.
    """
    S = S0
    for _ in range(y - 1):
        S = passo(S, delta)
    _, _, fita_final = decodificar_configuracao(S)
    return fita_final

## 3. Laço do Interpretador Universal: `μy [T(e, S0, y) = 0]`

In [14]:
def mu_busca_parada(delta: dict, S0: int, limite_seguranca: int = 2000) -> int | None:
    """
    μy [T(e, S0, y)]: busca pelo MENOR y tal que a máquina para no passo y.

    Esta é a ÚNICA operação de todo o simulador que NÃO é primitiva
    recursiva: não existe, em geral, uma cota a priori sobre y — não há
    como saber de antemão quantos passos uma máquina arbitrária levará
    até parar (ou se algum dia vai parar). É exatamente esta busca
    ilimitada que introduz o Problema da Parada e a indecidibilidade.

    'limite_seguranca' NÃO faz parte da definição matemática de μ — é
    apenas um teto artificial para que a demonstração em notebook não
    trave num loop real. Matematicamente, μy roda para sempre se a
    máquina nunca parar.
    """
    y = 1
    while y <= limite_seguranca:
        if T(delta, S0, y):
            return y
        y += 1
    return None  # limite de segurança atingido: não decidimos (ilustra a indecidibilidade)


def simular(delta: dict, fita_inicial: list[int], estado_inicial: int = 0,
            posicao_inicial: int = 0, limite_seguranca: int = 2000):
    """
    Função universal completa: f(x) = U(μy [T(e, x, y) = 0])

    Retorna (y_parada, fita_final) ou (None, None) se não decidiu parar
    dentro do limite de segurança.
    """
    S0 = codificar_configuracao(estado_inicial, posicao_inicial, fita_inicial)
    y = mu_busca_parada(delta, S0, limite_seguranca)
    if y is None:
        return None, None
    fita_final = U(delta, S0, y)
    return y, fita_final

## 4. Aplicações de Teste

### Teste 1 — Máquina que PARA em N passos (sucessor unário)

In [15]:
# Alfabeto: 0 = branco, 1 = marca unária
# Estado 0: percorre os 1's para a direita; ao achar branco, escreve 1
#           e vai para o estado 1 (sem transição definida -> halt).
delta_sucessor = {
    (0, 1): (0, 1, +1),   # continua andando enquanto lê 1
    (0, 0): (1, 1, 0),    # encontra o fim, escreve 1 extra, muda de estado
    # (1, *) : ausente de propósito -> qualquer símbolo no estado 1 para a máquina
}
fita_inicial = [1, 1, 1]  # representa o número unário 3
print(f"Fita inicial : {fita_inicial}  (unário para 3)")

y_parada, fita_final = simular(delta_sucessor, fita_inicial)
print(f"y (nº de passos até parar) = {y_parada}")
print(f"Fita final                 = {fita_final}  (unário para 4 = 3+1)")

Fita inicial : [1, 1, 1]  (unário para 3)
y (nº de passos até parar) = 5
Fita final                 = [1, 1, 1, 1]  (unário para 4 = 3+1)


**Verificação direta do predicado `T` (Kleene):**

In [16]:
S0 = codificar_configuracao(0, 0, fita_inicial)
for y_teste in range(1, y_parada + 3):
    resultado = T(delta_sucessor, S0, y_teste)
    marca = "  <-- MENOR y que satisfaz T (achado por μy)" if y_teste == y_parada else ""
    print(f"  T(e, S0, y={y_teste}) = {resultado}{marca}")

  T(e, S0, y=1) = False
  T(e, S0, y=2) = False
  T(e, S0, y=3) = False
  T(e, S0, y=4) = False
  T(e, S0, y=5) = True  <-- MENOR y que satisfaz T (achado por μy)
  T(e, S0, y=6) = False
  T(e, S0, y=7) = False


### Teste 2 — Máquina que ENTRA EM LOOP INFINITO

In [17]:
# Estado 0: para qualquer símbolo lido, reescreve o mesmo símbolo e
# NÃO se move — transição sempre definida, a máquina nunca para.
delta_loop = {
    (0, 0): (0, 0, 0),
    (0, 1): (0, 1, 0),
}
fita_loop = [1, 0, 1]
print(f"Fita inicial : {fita_loop}")
print("Buscando y com limite de segurança = 50 (μy NÃO teria cota real)...")

y_loop, fita_loop_final = simular(delta_loop, fita_loop, limite_seguranca=50)
if y_loop is None:
    print("μy NÃO encontrou parada dentro do limite de segurança.")
    print("Isso ilustra o Problema da Parada: sem cota a priori, a busca")
    print("por μy pode, em geral, rodar indefinidamente — é exatamente")
    print("essa a operação que está FORA da classe FRP.")
else:
    print(f"(inesperado) y = {y_loop}, fita = {fita_loop_final}")

Fita inicial : [1, 0, 1]
Buscando y com limite de segurança = 50 (μy NÃO teria cota real)...
μy NÃO encontrou parada dentro do limite de segurança.
Isso ilustra o Problema da Parada: sem cota a priori, a busca
por μy pode, em geral, rodar indefinidamente — é exatamente
essa a operação que está FORA da classe FRP.


## Conclusão

Toda a lógica de fita/transição (`passo`, `T`, `U`) é **100% Recursiva
Primitiva** — sempre decide e sempre termina para `y` fixo. A única
etapa que precisa do operador `μ` (minimização **não** limitada) é a
busca pelo menor `y` de parada — e é justamente aí que mora a
indecidibilidade do Problema da Parada (Teorema de Kleene).

Note também que o Projeto 05 inteiro reaproveitou, sem reescrever nada,
as três funções centrais do Projeto 04 (`codificar`, `tamanho`,
`obter_elemento`) — a mesma ferramenta de codificação usada para
comprimir uma lista `[3, 1, 4]` num natural é usada aqui, em camadas,
para comprimir fita + estado + posição de uma Máquina de Turing inteira
num único natural.